# Digital Echoes vs. Bed Nights
## Predicting Viral Tourism Demand Shocks from Digital Signals

**DTU 42578 Advanced Business Analytics — Spring 2026**  
*Group: Alex (Spain · ES), Beatriz (Croatia · HR), Cesia (Italy · IT & Albania · AL), Frede (Malta · MT & Data Engineering)*

---

## 1. Introduction

Viral social-media content can trigger rapid, hard-to-predict surges in tourism demand — so-called *viral tourism shocks*. This report investigates whether **digital attention signals** (Google Trends, Wikipedia pageviews) measured *before* a surge carry sufficient predictive power to forecast Eurostat bed-night arrivals at the regional level across six EU destinations: Spain, Croatia, Italy, Albania, Malta, and Greece.

**Research question:** Do digital signals predict tourist arrivals with a measurable lead time, and can we build a resilience-aware system to help destinations prepare?

**Methods (≥ 3 course topics):**
| Method | Course topic |
|--------|-------------|
| Random Forest + SHAP | Forecasting · Explainable AI |
| Graph Neural Network (GNN) | GNNs — spatial spillover |
| Recommender System | Recommender systems |
| Granger causality + COVID policy controls | Causal methods · Endogeneity |
| Seasonality decomposition (month + YoY) | Time-series methods |

---
## 2. Data Collection

### 2.0 Setup & Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})

# ── Paths ────────────────────────────────────────────────────────────────────
# Colab: mount Drive first, then set DATA_DIR to your Drive path.
# Local: notebook lives in group_repo/, data is at ../data/processed/
if os.path.exists('/content/drive'):
    DATA_DIR = '/content/drive/MyDrive/viral_tourism/data/processed'
else:
    DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'data', 'processed')
    if not os.path.exists(DATA_DIR):  # fallback: notebook run from project root
        DATA_DIR = os.path.join(os.getcwd(), 'data', 'processed')

MASTER_FILE = os.path.join(DATA_DIR, 'master_tourism_dataset.csv')
OXCGRT_FILE = os.path.join(DATA_DIR, 'oxcgrt_covid_data.csv')

print('Data directory:', DATA_DIR)
print('Master file exists:', os.path.exists(MASTER_FILE))


### 2.1 Datasets Used

| Dataset | Source | Coverage | Key variables |
|---------|--------|----------|---------------|
| Eurostat tourist nights spent (tour_occ_nin2m) | Eurostat (Excel download) | EU NUTS2 regions, 2020–2026 | `nights_spent_nuts2`, `nights_spent_country` |
| Eurostat monthly arrivals (tour_occ_arm) | Eurostat API | Country level, 2020–2026 | `arrivals_country` |
| Eurostat bed places (tour_cap_nat) | Eurostat API | Country level, annual | `bed_places_country` |
| Google Trends | pytrends (unofficial API) | 30 cities, worldwide normalisation, 2020–2026 | `gt_airbnb`, `gt_hotel`, `gt_flights`, `gt_attraction_1/2/3` |
| OxCGRT COVID policy (v4) | Oxford COVID-19 Government Response Tracker | Country-level, 2020–2022 | `C6M_stay_at_home`, `C4M_gathering_restrictions`, `C8EV_travel_controls` |

**Destinations covered:**  
Spain (ES — 6 cities, 6 NUTS2 regions), Croatia (HR — 5 cities, 4 NUTS2 regions), Italy (IT — 5 cities, 5 NUTS2 regions), Albania (AL — 5 cities, 3 NUTS2 regions), Malta (MT — 4 cities, **1 NUTS2 region**), Greece (GR — 5 cities, 5 NUTS2 regions).  
**Total: 2,250 rows × 16 columns** (75 months × 30 cities).


### 2.2 APIs & Data Acquisition

**Google Trends — pytrends**  
Data collected via `pytrends.TrendReq` with worldwide geo-filter (`geo=''`) to ensure comparability across countries. Queries batched in groups of 5 with an anchor term for cross-batch normalisation. *Note: urllib3 v2 requires a monkey-patch (`method_whitelist` removal) before importing `TrendReq`.*

**Eurostat — SDMX REST API**  
Dataset `tour_occ_nin2m` (monthly nights spent by residents and non-residents, NUTS2 level). Downloaded as TSV and reshaped via `cleaning_data.py`.

**Wikipedia — Wikimedia REST API**  
`https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/…` — daily pageviews aggregated to monthly.

**OxCGRT — direct CSV download**  
Oxford's `OxCGRT_compact_national_v1.csv` filtered to indicators C4M, C6M, C8EV and aggregated to monthly means.

### 2.3 Data Pipeline Architecture

> **TODO (Frede):** Insert architecture diagram here (the drawing you made showing the full pipeline from raw APIs → master dataset → feature engineering → models).

<!-- Embed as: ![Pipeline architecture](images/architecture.png) -->

### 2.4 Final Dataset

In [ ]:
df = pd.read_csv(MASTER_FILE, parse_dates=['date'])
df.sort_values(['country', 'region', 'city', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Shape: {df.shape}')
print(f'Countries: {sorted(df["country"].unique())}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## 3. Dataset Pre-processing & Visualisation

### 3.1 Seasonality Features

Three features capture the seasonal structure of tourism demand:  
- `month` — calendar month (1–12) used as a cyclic control  
- `arrivals_last_year` — 12-month lag of `arrivals_country` within each geographic unit  
- `arrivals_yoy_pct_change` — year-over-year percentage change  

*Script adapted from `FINAL/add_seasonality.py` (Frede's branch).*

In [ ]:
GEO_KEYS = ['country', 'region', 'city']

# Month
df['month'] = df['date'].dt.month

# 12-month lag within each geographic unit
df['arrivals_last_year'] = (
    df.groupby(GEO_KEYS)['arrivals_country'].shift(12)
)

# Year-over-year % change
df['arrivals_yoy_pct_change'] = (
    (df['arrivals_country'] - df['arrivals_last_year'])
    / df['arrivals_last_year'].replace(0, np.nan)
    * 100
)

print('Seasonality features added.')
print(df[['date', 'country', 'city', 'month', 'arrivals_country',
          'arrivals_last_year', 'arrivals_yoy_pct_change']].dropna().head(10))

### 3.2 Endogeneity Controls — COVID Policy Stringency

To address endogeneity between digital signals and actual arrivals during the pandemic, we merge three OxCGRT indicators as control variables:

| Feature | OxCGRT indicator | Interpretation |
|---------|-----------------|----------------|
| `dest_stay_at_home` | C6M | Destination stay-at-home stringency |
| `dest_gathering_restrictions` | C4M | Restrictions on gatherings at destination |
| `origin_top3_travel_ban_avg` | C8EV (avg. top-3 origin countries) | International travel ban from main sending markets |

*Script adapted from `FINAL/build_dataset_custom_covid.py` (Frede's branch).*

In [ ]:
import os

TOP_ORIGINS = {
    'Albania':  ['Kosovo', 'North Macedonia', 'Italy'],
    'Croatia':  ['Germany', 'Slovenia', 'Austria'],
    'Greece':   ['United Kingdom', 'Germany', 'France'],
    'Italy':    ['Germany', 'United States', 'United Kingdom'],
    'Malta':    ['United Kingdom', 'Italy', 'France'],
    'Spain':    ['United Kingdom', 'France', 'Germany'],
}

if not os.path.exists(OXCGRT_FILE):
    print(f'[SKIP] OxCGRT file not found at {OXCGRT_FILE}. '
          'Download oxcgrt_covid_data.csv and place it in DATA_DIR to run this section.')
else:
    ox = pd.read_csv(
        OXCGRT_FILE,
        usecols=['CountryName', 'Date',
                 'C8EV_International travel controls',
                 'C6M_Stay at home requirements',
                 'C4M_Restrictions on gatherings'],
        low_memory=False,
    )
    ox['Date'] = pd.to_datetime(ox['Date'].astype(str), format='%Y%m%d')
    ox['year_month'] = ox['Date'].values.astype('datetime64[M]')
    ox_m = ox.groupby(['CountryName', 'year_month'], as_index=False).mean(numeric_only=True)

    # Destination controls
    dest_ox = ox_m[['CountryName', 'year_month',
                    'C6M_Stay at home requirements',
                    'C4M_Restrictions on gatherings']].rename(columns={
        'C6M_Stay at home requirements':  'dest_stay_at_home',
        'C4M_Restrictions on gatherings': 'dest_gathering_restrictions',
    })
    df = df.merge(dest_ox, how='left',
                  left_on=['country', 'date'],
                  right_on=['CountryName', 'year_month'])
    df.drop(columns=['CountryName', 'year_month'], inplace=True)

    # Origin travel-ban average (top 3 sending countries)
    travel_lkp = ox_m.set_index(['CountryName', 'year_month'])[
        'C8EV_International travel controls'].to_dict()

    def _origin_ban_avg(row):
        origins = TOP_ORIGINS.get(row['country'], [])
        vals = [travel_lkp.get((o, row['date'])) for o in origins]
        vals = [v for v in vals if v is not None]
        return sum(vals) / len(vals) if vals else np.nan

    df['origin_top3_travel_ban_avg'] = df.apply(_origin_ban_avg, axis=1)

    # Fill pre/post-pandemic NaNs with 0
    covid_cols = ['dest_stay_at_home', 'dest_gathering_restrictions', 'origin_top3_travel_ban_avg']
    df[covid_cols] = df[covid_cols].fillna(0)

    print('COVID policy controls merged.')
    print(df[['date', 'country', 'city'] + covid_cols].dropna().head(8))

### 3.3 Exploratory Data Analysis & Visualisation

> Each team member adds plots for their own region(s) below.

In [ ]:
# ── Spain — EDA (Alex) ────────────────────────────────────────────────────────
# TODO (Alex): time-series plot of GT signals vs. nights_spent for ES regions,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass

In [ ]:
# ── Croatia — EDA (Beatriz) ───────────────────────────────────────────────────
# TODO (Beatriz): time-series, shock markers, correlation heatmap for HR.
pass

In [ ]:
# ── Italy — EDA (Cesia) ───────────────────────────────────────────────────────
# TODO (Cesia): time-series, shock markers, correlation heatmap for IT.
pass

In [ ]:
# ── Albania — EDA (Cesia) ─────────────────────────────────────────────────────
# TODO (Cesia): time-series, shock markers, correlation heatmap for AL.
pass

In [ ]:
# ── Malta — EDA (Frede) ───────────────────────────────────────────────────
# Malta has 4 cities but ONE NUTS2 region — aggregate before plotting.
MT_GT_COLS = ['gt_airbnb','gt_hotel','gt_flights','gt_attraction_1','gt_attraction_2','gt_attraction_3']
df_malta_region = (
    df[df['country']=='Malta']
    .groupby('date', as_index=False)
    .agg({**{c: 'mean' for c in MT_GT_COLS},
          'nights_spent_nuts2': 'first',
          'arrivals_country':   'first',
          'month': 'first'})
)
# TODO (Frede): time-series plot of GT signals vs. nights_spent_nuts2 for Malta,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass


In [ ]:
# ── Greece — EDA (Alex / Beatriz) ────────────────────────────────────────
# TODO: time-series plot of GT signals vs. nights_spent_nuts2 for GR cities,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass


---
## 4. Prediction

### 4.1 Why Random Forest?

> **TODO (all):** Justify the model choice here — why Random Forest over linear regression, ARIMA, or gradient boosting for this setting? Consider: non-linearity, mixed feature types (GT signals + seasonal + policy controls), interpretability via SHAP, robustness to missing values.

### 4.2 Spain (Alex — ES)

In [ ]:
# ── Spain — Random Forest + SHAP ────────────────────────────────────────
# TODO (Spain):
#   1. Filter: df_spain = df[df['country']=='Spain']
#   2. Features X: GT signals (gt_airbnb, gt_hotel, gt_flights,
#      gt_attraction_1/2/3), month, arrivals_last_year,
#      arrivals_yoy_pct_change, covid controls if present
#   3. Target y: nights_spent_nuts2  ← primary target (NUTS2 regional level)
#   4. Time-based train/test split (last 12 months = test)
#   5. Fit RandomForestRegressor(n_estimators=200, random_state=42)
#   6. MAE, R² — print and store in results dict for section 4.8
#   7. Plot actual vs. predicted
#   8. SHAP: TreeExplainer → summary_plot + bar plot of mean |SHAP|
pass


### 4.3 Croatia (Beatriz — HR)

In [ ]:
# ── Croatia — Random Forest + SHAP ────────────────────────────────────────
# TODO (Croatia):
#   1. Filter: df_croatia = df[df['country']=='Croatia']
#   2. Features X: GT signals (gt_airbnb, gt_hotel, gt_flights,
#      gt_attraction_1/2/3), month, arrivals_last_year,
#      arrivals_yoy_pct_change, covid controls if present
#   3. Target y: nights_spent_nuts2  ← primary target (NUTS2 regional level)
#   4. Time-based train/test split (last 12 months = test)
#   5. Fit RandomForestRegressor(n_estimators=200, random_state=42)
#   6. MAE, R² — print and store in results dict for section 4.8
#   7. Plot actual vs. predicted
#   8. SHAP: TreeExplainer → summary_plot + bar plot of mean |SHAP|
pass


### 4.4 Italy (Cesia — IT)

In [ ]:
# ── Italy — Random Forest + SHAP ────────────────────────────────────────
# TODO (Italy):
#   1. Filter: df_italy = df[df['country']=='Italy']
#   2. Features X: GT signals (gt_airbnb, gt_hotel, gt_flights,
#      gt_attraction_1/2/3), month, arrivals_last_year,
#      arrivals_yoy_pct_change, covid controls if present
#   3. Target y: nights_spent_nuts2  ← primary target (NUTS2 regional level)
#   4. Time-based train/test split (last 12 months = test)
#   5. Fit RandomForestRegressor(n_estimators=200, random_state=42)
#   6. MAE, R² — print and store in results dict for section 4.8
#   7. Plot actual vs. predicted
#   8. SHAP: TreeExplainer → summary_plot + bar plot of mean |SHAP|
pass


### 4.5 Albania (Cesia — AL)

In [ ]:
# ── Albania — Random Forest + SHAP ────────────────────────────────────────
# TODO (Albania):
#   1. Filter: df_albania = df[df['country']=='Albania']
#   2. Features X: GT signals (gt_airbnb, gt_hotel, gt_flights,
#      gt_attraction_1/2/3), month, arrivals_last_year,
#      arrivals_yoy_pct_change, covid controls if present
#   3. Target y: nights_spent_nuts2  ← primary target (NUTS2 regional level)
#   4. Time-based train/test split (last 12 months = test)
#   5. Fit RandomForestRegressor(n_estimators=200, random_state=42)
#   6. MAE, R² — print and store in results dict for section 4.8
#   7. Plot actual vs. predicted
#   8. SHAP: TreeExplainer → summary_plot + bar plot of mean |SHAP|
pass


### 4.6 Malta (Frede — MT)

In [ ]:
# ── Malta — Random Forest + SHAP (Frede) ─────────────────────────────────
# Malta = ONE region. Aggregate all 4 cities before modelling.
MT_GT_COLS = ['gt_airbnb','gt_hotel','gt_flights','gt_attraction_1','gt_attraction_2','gt_attraction_3']
df_malta_model = (
    df[df['country']=='Malta']
    .groupby('date', as_index=False)
    .agg({**{c: 'mean' for c in MT_GT_COLS},
          'nights_spent_nuts2': 'first',
          'arrivals_country':   'first',
          'month': 'first',
          **({col: 'first' for col in
              ['dest_stay_at_home','dest_gathering_restrictions','origin_top3_travel_ban_avg']
              if col in df.columns})})
)
df_malta_model['arrivals_last_year'] = df_malta_model['nights_spent_nuts2'].shift(12)
df_malta_model['arrivals_yoy_pct_change'] = (
    (df_malta_model['nights_spent_nuts2'] - df_malta_model['arrivals_last_year'])
    / df_malta_model['arrivals_last_year'].replace(0, np.nan) * 100
)
df_malta_model = df_malta_model.dropna()

# TODO (Frede): define FEATURES, TARGET = 'nights_spent_nuts2',
# time-based train/test split, fit RandomForestRegressor,
# compute MAE + R², plot actual vs. predicted, SHAP summary_plot.
pass


### 4.7 Greece (Alex / Beatriz — GR)


In [ ]:
# ── Greece — Random Forest + SHAP ────────────────────────────────────────
# TODO: same pipeline as Spain, filtered to country == 'Greece'.
# Target: nights_spent_nuts2 (NUTS2 region level — 5 regions).
# Features: GT signals, month, arrivals_last_year, arrivals_yoy_pct_change,
#           covid controls if available.
pass


### 4.8 Mean Prediction Comparison Across Regions


In [ ]:
# ── Cross-region comparison (all) ────────────────────────────────────────────
# TODO (all): Collect per-region MAE and R² into a summary DataFrame and
# plot a bar chart comparing model performance across all countries.
# Also plot mean predicted vs. mean actual arrivals per country.
#
# Example structure:
# results = {
#     'Spain':   {'mae': ..., 'r2': ...},
#     'Croatia': {'mae': ..., 'r2': ...},
#     ...   # add each country once prediction sections above are filled
# }
pass

---
## 5. Graph Neural Network (GNN)

### 5.1 Why a GNN?

> **TODO:** Justify the GNN — viral tourism shocks don't respect national borders. A destination that goes viral (e.g. Albania via TikTok) can trigger spillover demand to neighbouring regions. A GNN allows us to model these spatial dependencies explicitly by representing destinations as nodes and shared tourist-flow corridors as edges, enabling risk propagation and failure prediction across the network.
>
> Describe graph construction: nodes = destinations, edges = shared origin markets / geographic proximity, edge weights = shared visitor nationality share.

In [ ]:
# ── GNN implementation ────────────────────────────────────────────────────────
# TODO: Implement graph construction + GNN training here.
# Suggested libraries: torch_geometric or spektral (Keras).
pass

---
## 6. Recommender System

### 6.1 Concept & Justification

> **TODO:** Explain the recommender system — what does it recommend, to whom, and why is it the right tool here? Possible framing: given a destination's current digital-signal profile, recommend which *resilience interventions* (capacity limits, diversification campaigns, off-season promotion) are most likely to dampen an incoming demand shock, based on similarity to historical shock-recovery patterns from other destinations.

In [ ]:
# ── Recommender System implementation ─────────────────────────────────────────
# TODO: Implement recommender logic here.
pass

---
## 7. Final Solution System Architecture

> **TODO (all):** Insert the end-to-end system architecture diagram showing how all components connect:
> Digital signals (GT / Wiki) → Feature engineering (seasonality + COVID controls) → Random Forest per region → GNN spillover layer → Recommender System → Resilience dashboard output.

<!-- ![Final architecture](images/final_architecture.png) -->

---
## 8. Conclusion

> **TODO (all):** Summarise main findings:
> - Do GT signals predict arrivals? With what lead time?
> - Which features matter most (SHAP)?
> - What do the GNN spillover results reveal?
> - How useful is the recommender for resilience planning?
>
> Then list limitations and future work.